In [1]:
import sys
!{sys.executable} -m pip install "numpy<2" --force-reinstall

  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [2]:
import os
import sys

# --- 1. EMERGENCY DLL PATHING (The fix for shm.dll) ---
env_path = r"c:\Users\User\anaconda3\envs\cineiq"
dll_path = os.path.join(env_path, "Lib", "site-packages", "torch", "lib")
bin_path = os.path.join(env_path, "Library", "bin")

if os.path.exists(dll_path):
    os.add_dll_directory(dll_path)
if os.path.exists(bin_path):
    os.add_dll_directory(bin_path)

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# --- 2. IMPORT THE STACK (Now with NumPy 1.26.4) ---
import numpy as np
import torch
import pandas as pd
import sklearn
import surprise
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# --- 3. THE FINAL TEST ---
print(f"NumPy Version: {np.__version__}") # Should be 1.26.4
print(f"Torch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

print("\nEvery single CINEIQ package successfully loaded with zero errors!")

NumPy Version: 1.26.4
Torch Version: 2.3.1+cu118
CUDA Available: True

Every single CINEIQ package successfully loaded with zero errors!


In [3]:
print("INITIATING SYSTEM CHECK...\n")

# 1. The Data & Math Stack
print("1/4 Loading Data Stack...")
import numpy as np
import pandas as pd
print(" NumPy & Pandas loaded.")

# 2. The Traditional ML Stack
print("2/4 Loading ML Stack...")
import sklearn
import surprise
print(" Scikit-Learn & Scikit-Surprise loaded.")

# 3. The Web & Logging Stack
print("3/4 Loading Web Stack...")
import fastapi
import streamlit as st
import requests
import mlflow
import plotly
print(" FastAPI, Streamlit, MLflow, Plotly loaded.")

# 4. The NLP & Hardware Acceleration Stack
print("4/4 Loading Deep Learning & GPU Stack...\n")
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import torch
from transformers import pipeline

# ACTIVE TESTS 

# A. VADER Test
vader_analyzer = SentimentIntensityAnalyzer()
vader_score = vader_analyzer.polarity_scores("The backend is finally working!")
print(f" VADER Baseline Online: {vader_score}")

# B. PyTorch CUDA Test
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f" HARDWARE ACCELERATION ACTIVE: {gpu_name} detected.")
    device_id = 0
else:
    print(" WARNING: GPU not detected. PyTorch will use CPU.")
    device_id = -1

# C. Hugging Face Test (This might take a few seconds to download the tiny model the first time)
try:
    print("\n  Spinning up DistilBERT on hardware...")
    sentiment_tester = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device_id) 
    hf_result = sentiment_tester("Building this recommendation engine is a massive success!")
    print(f"Hugging Face Engine Online: {hf_result}")
except Exception as e:
    print(f" Hugging Face Error: {e}")
print("\n CINEIQ ARCHITECTURE IS FULLY OPERATIONAL")



INITIATING SYSTEM CHECK...

1/4 Loading Data Stack...
 NumPy & Pandas loaded.
2/4 Loading ML Stack...
 Scikit-Learn & Scikit-Surprise loaded.
3/4 Loading Web Stack...


c:\Users\User\anaconda3\envs\cineiq\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\User\anaconda3\envs\cineiq\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 FastAPI, Streamlit, MLflow, Plotly loaded.
4/4 Loading Deep Learning & GPU Stack...

 VADER Baseline Online: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
 HARDWARE ACCELERATION ACTIVE: NVIDIA GeForce RTX 3050 6GB Laptop GPU detected.

  Spinning up DistilBERT on hardware...


Device set to use cuda:0


Hugging Face Engine Online: [{'label': 'POSITIVE', 'score': 0.9998646974563599}]

 CINEIQ ARCHITECTURE IS FULLY OPERATIONAL


c:\Users\User\anaconda3\envs\cineiq\lib\site-packages\transformers\models\distilbert\modeling_distilbert.py:392: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [4]:
gscores = pd.read_csv('genome-scores.csv')
gtags = pd.read_csv('genome-tags.csv')
links = pd.read_csv('links.csv')
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
tags = pd.read_csv('tags.csv')

In [5]:
user_counts = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()
#print(user_counts)
#print(movie_counts)

In [6]:
# Keep only the users who have rated 50 or more movies
# This ensures the model learns from users with established tastes
ratings_filtered = ratings[ratings['userId'].isin(user_counts[user_counts >= 50].index)]

# Optional: Do the same for movies to ensure every movie has enough data
movie_counts = ratings_filtered['movieId'].value_counts()
ratings_filtered = ratings_filtered[ratings_filtered['movieId'].isin(movie_counts[movie_counts >= 50].index)]

In [7]:
from surprise import Dataset, Reader

# Step 3a: Initialize the Reader with the MovieLens rating scale
# This tells the library exactly what the min and max possible scores are.
reader = Reader(rating_scale=(0.5, 5.0))

# Step 3b: Load the DataFrame into the Surprise Dataset format
# You MUST provide the columns in this exact order: [User, Item, Rating]
data = Dataset.load_from_df(ratings_filtered[['userId', 'movieId', 'rating']], reader)

In [8]:
from surprise.model_selection import train_test_split

# Split the data: 80% for the model to learn from, 20% for testing its "guesses"
# random_state=42 ensures you get the same split every time you run the code
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [9]:
from surprise import SVD

# Step 5a: Initialize the SVD algorithm
# n_factors: The number of hidden traits (e.g., genre affinity, tone) to identify
# lr_all: The learning rate (how fast the model adjusts its weights)
# reg_all: The regularization term (prevents the model from memorizing/overfitting)
algo = SVD(n_factors=100, lr_all=0.005, reg_all=0.02)

# Step 5b: Train the model on the training set
# This process fills in the gaps in the matrix to enable future predictions
algo.fit(trainset)

In [10]:
from surprise import accuracy
import mlflow

# Step 6a: Tell the model to "guess" the ratings for the 20% we hid earlier
predictions = algo.test(testset)

# Step 6b: Start an MLflow run to record the results [cite: 127]
with mlflow.start_run(run_name="SVD_Base_Model"):
    # Calculate the error metrics
    rmse_value = accuracy.rmse(predictions)
    mae_value = accuracy.mae(predictions)
    
    # Log the metrics to your experiment tracker
    mlflow.log_metric("rmse", rmse_value)
    mlflow.log_metric("mae", mae_value)

2026/04/27 21:58:09 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



RMSE: 0.7655
MAE:  0.5805


In [11]:
#import joblib

# Save the trained SVD algorithm as a Python Pickle (.pkl) file
# This file is a 'Model Artifact' that will be used by the FastAPI server
#joblib.dump(algo, 'svd_model.pkl')

#print("Final Step Complete: SVD model artifact exported as 'svd_model.pkl'.")